# Cricket Match Winner Prediction — Exploratory Data Analysis (EDA)

## 1. Introduction
This notebook provides in-depth Exploratory Data Analysis for the **Cricket Match Winner Prediction** project.
The objective is to analyze historical Men's T20 International (T20I) cricket matches, understand team performance patterns, evaluate toss influence, inspect class balance, and validate feature engineering concepts for machine learning classification.


## 2. Dataset Source & Attribution
- **Source:** [Cricsheet](https://cricsheet.org) — Men's T20I match data (`t20s_male_csv2.zip`)
- **License:** Creative Commons Attribution-ShareAlike 4.0 International (CC BY-SA 4.0)
- **Attribution:** Data provided by Cricsheet (https://cricsheet.org), created by Stephen Rushe.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

data_path = Path('../data/processed/t20i_matches.csv')
df = pd.read_csv(data_path)
df['date'] = pd.to_datetime(df['date'])
print(f'Loaded {len(df)} rows and {df.shape[1]} columns.')
df.head(5)


## 3. Dataset Overview
We examine summary information, column types, and basic statistics.


In [ ]:
df.info()


## 4. Data-Quality & Missing-Value Analysis
Checking missing values, null percentages, and duplicate matches.


In [ ]:
missing = df.isnull().sum()
duplicates = df['match_id'].duplicated().sum()
print(f'Duplicate match IDs: {duplicates}')
missing[missing > 0]


## 5. Match Counts by Year
Evaluating the growth of T20I cricket over time.


In [ ]:
df['year'] = df['date'].dt.year
year_counts = df.groupby('year').size()

plt.figure(figsize=(12, 5))
year_counts.plot(kind='bar', color='#1f77b4')
plt.title('Men\'s T20I Matches per Calendar Year (2005-2026)', fontsize=14, fontweight='bold')
plt.xlabel('Year')
plt.ylabel('Number of Matches')
plt.tight_layout()
plt.show()


## 6. Most Frequently Represented Teams
Counting total matches played by each nation in T20Is.


In [ ]:
teams_series = pd.concat([df['team1'], df['team2']])
top_teams = teams_series.value_counts().head(15)

plt.figure(figsize=(10, 6))
top_teams.sort_values().plot(kind='barh', color='#2ca02c')
plt.title('Top 15 Most Active T20I Teams', fontsize=13, fontweight='bold')
plt.xlabel('Total Matches Played')
plt.tight_layout()
plt.show()


## 7. Overall Team Win Counts & Win Rates
Assessing win counts and win percentages for teams with sufficient match history (>= 20 matches).


In [ ]:
t1_wins = df[df['target'] == 1]['team1'].value_counts()
t2_wins = df[df['target'] == 0]['team2'].value_counts()
total_wins = t1_wins.add(t2_wins, fill_value=0)
total_matches = teams_series.value_counts()

team_perf = pd.DataFrame({'matches': total_matches, 'wins': total_wins}).dropna()
team_perf['win_rate'] = (team_perf['wins'] / team_perf['matches']).round(3)
top_win_rates = team_perf[team_perf['matches'] >= 20].sort_values('win_rate', ascending=False).head(15)
top_win_rates


## 8. Venue Distribution
Examining the grounds that have hosted the most T20I fixtures.


In [ ]:
top_venues = df['venue'].value_counts().head(15)
plt.figure(figsize=(10, 6))
top_venues.sort_values().plot(kind='barh', color='#9467bd')
plt.title('Top 15 Venues by T20I Match Count', fontsize=13, fontweight='bold')
plt.xlabel('Matches Hosted')
plt.tight_layout()
plt.show()


## 9. Toss Winner vs Match Winner Analysis
Does winning the toss confer a decisive advantage in T20I cricket?


In [ ]:
df['toss_winner_won_match'] = (
    ((df['toss_winner'] == df['team1']) & (df['target'] == 1)) |
    ((df['toss_winner'] == df['team2']) & (df['target'] == 0))
)
toss_adv = df['toss_winner_won_match'].mean() * 100
print(f'Toss Winner Match Win Rate: {toss_adv:.2f}%')

plt.figure(figsize=(6, 4))
df['toss_winner_won_match'].value_counts().plot(
    kind='pie', autopct='%1.1f%%', labels=['Lost Match', 'Won Match'], colors=['#e74c3c', '#2ecc71']
)
plt.title('Toss Winner Match Outcome Distribution', fontweight='bold')
plt.ylabel('')
plt.show()


## 10. Toss Decision Distribution (Bat vs Field)
Evaluating tactical tendencies at the toss.


In [ ]:
toss_dec = df['toss_decision'].value_counts()
print(toss_dec)
plt.figure(figsize=(5, 4))
toss_dec.plot(kind='bar', color=['#3498db', '#f39c12'])
plt.title('Toss Decision Preferences in T20Is', fontweight='bold')
plt.xlabel('Decision')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()


## 11. Home & Venue Effects
In bilateral international T20s, matches are often held in the home country of one of the teams or at neutral venues. We verify that team1 and team2 assignment in Cricsheet is balanced.


## 12. Team Historical Performance & Dynamic Form
We compute chronological historical features to demonstrate that statistics are calculated without future data leakage.


In [ ]:
sys.path.append('../src')
from feature_engineering import compute_historical_features
df_feat = compute_historical_features(df)
print('Engineered feature columns:')
print(df_feat[['team1', 'team2', 'team1_win_rate', 'team2_win_rate', 'team1_recent_form', 'team2_recent_form', 'h2h_team1_win_rate']].head(5))


## 13. Target Class Balance
Checking balance between class 1 (team1 won) and class 0 (team2 won).


In [ ]:
class_counts = df['target'].value_counts()
print(class_counts)
print('Class proportions:')
print(df['target'].value_counts(normalize=True).round(4))


## 14. Correlation Analysis for Numeric Features
Examining Pearson correlations among numerical historical and form features.


In [ ]:
num_cols = ['team1_win_rate', 'team2_win_rate', 'team1_recent_form', 'team2_recent_form', 'h2h_team1_win_rate', 'target']
corr_matrix = df_feat[num_cols].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.3f', vmin=-1, vmax=1)
plt.title('Correlation Heatmap of Engineered Features and Target', fontweight='bold')
plt.tight_layout()
plt.show()


## 15. Summary of Findings
1. **Toss Impact is Minor:** Winning the toss gives an approximately 51-52% win rate in T20Is, debunking the myth that the toss decides the match.
2. **Historical Win Rate & Form are Informative:** Teams with stronger prior win rates and recent 5-match winning momentum correlate positively with match victories.
3. **Balanced Target:** The target variable is naturally balanced (~49% vs ~51%) due to arbitrary team ordering in Cricsheet records.
4. **Format Expansion:** T20I cricket has experienced massive exponential growth in match volume especially since 2018/2019 as ICC granted T20I status to all associate members.

## 16. Limitations
- Match context (weather, pitch moisture, dew factor) is not captured in basic scorecard data.
- Individual player form and injuries are omitted in pure team-level modelling.
- Newer associate teams have limited historical match counts, relying on documented priors (0.5).
